# RecVis Assignment 3 — Baselines

Train on labeled CIFAR-10 and submit predictions for the unlabeled diffusion-hard test set. **Validation accuracy is measured on ordinary CIFAR-10 and is not expected to predict Kaggle accuracy.**

Attach the competition data through Kaggle's **Add Input** panel and enable a GPU. On Kaggle, update the default input path if needed. For local execution, set the `RECVIS_INPUT_DIR` and optional `RECVIS_OUTPUT_DIR` environment variables.

In [1]:
# Kaggle already includes PyTorch. Install this only for the optional CLIP cell.
# !pip install -q open-clip-torch

from pathlib import Path
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as T
from PIL import Image
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset

# Prefer RECVIS_* env vars, then local project data/outputs, then Kaggle defaults.
KAGGLE_INPUT = Path('/kaggle/input/recvis-mva-a3-2026')
KAGGLE_OUTPUT = Path('/kaggle/working')
REQUIRED = ('train.npz', 'validation.npz', 'test.npz', 'sample_submission.csv')

def _has_competition_files(folder):
    return all((folder / name).is_file() for name in REQUIRED)

def _find_local_input():
    here = Path.cwd().resolve()
    candidates = [here / 'data', here]
    candidates.extend(parent / 'data' for parent in here.parents)
    for candidate in candidates:
        if _has_competition_files(candidate):
            return candidate
    return None

def resolve_input_dir():
    if 'RECVIS_INPUT_DIR' in os.environ:
        return Path(os.environ['RECVIS_INPUT_DIR'])
    local = _find_local_input()
    return local if local is not None else KAGGLE_INPUT

def resolve_output_dir(input_dir):
    if 'RECVIS_OUTPUT_DIR' in os.environ:
        return Path(os.environ['RECVIS_OUTPUT_DIR'])
    if input_dir.name == 'data':
        return input_dir.parent / 'outputs'
    if _has_competition_files(Path.cwd() / 'data') or (Path.cwd() / 'main_notebook.ipynb').exists():
        return Path.cwd() / 'outputs'
    return KAGGLE_OUTPUT

INPUT_DIR = resolve_input_dir()
OUTPUT_DIR = resolve_output_dir(INPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42
CLASS_NAMES = ('airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
missing_files = sorted(name for name in REQUIRED if not (INPUT_DIR / name).is_file())
if missing_files:
    raise FileNotFoundError(f'Missing competition files in {INPUT_DIR}: {missing_files}')
print(f'Using {DEVICE}; input: {INPUT_DIR}')
print(f'output: {OUTPUT_DIR}')
print(sorted(path.name for path in INPUT_DIR.iterdir()))

Using cuda; input: C:\Users\aimin\Documents\ENS MVA\computer vision\TP3\recvis26_a3\data
output: C:\Users\aimin\Documents\ENS MVA\computer vision\TP3\recvis26_a3\outputs
['sample_submission.csv', 'test.npz', 'train.npz', 'validation.npz']


## Load the data

`train.npz` and `validation.npz` are labeled CIFAR-10. `test.npz` contains only image arrays and identifiers: it intentionally has no labels.

In [2]:
class NpzDataset(Dataset):
    def __init__(self, path, transform, labeled):
        with np.load(path) as archive:
            self.images = archive['images'].astype(np.uint8)
            self.labels = archive['labels'].astype(np.int64) if 'labels' in archive else None
            self.ids = archive['ids'].astype(np.int64) if 'ids' in archive else np.arange(len(self.images))
        if labeled and self.labels is None:
            raise ValueError(f'{path} must have labels')
        self.transform, self.labeled = transform, labeled
    def __len__(self): return len(self.images)
    def __getitem__(self, index):
        image = self.transform(Image.fromarray(self.images[index], mode='RGB'))
        return image, int(self.labels[index] if self.labeled else self.ids[index])

CIFAR_MEAN, CIFAR_STD = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)
def transforms_for(image_size=32, pretrained=False):
    mean, std = ((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)) if pretrained else (CIFAR_MEAN, CIFAR_STD)
    if image_size == 32:
        train_tf = T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip(), T.ToTensor(), T.Normalize(mean, std)])
        eval_tf = T.Compose([T.ToTensor(), T.Normalize(mean, std)])
    else:
        train_tf = T.Compose([T.RandomResizedCrop(image_size, scale=(0.8, 1.0)), T.RandomHorizontalFlip(), T.ToTensor(), T.Normalize(mean, std)])
        eval_tf = T.Compose([T.Resize((image_size, image_size)), T.ToTensor(), T.Normalize(mean, std)])
    return train_tf, eval_tf

def loaders(image_size=32, pretrained=False, batch_size=128, workers=0):
    train_tf, eval_tf = transforms_for(image_size, pretrained)
    train_set = NpzDataset(INPUT_DIR / 'train.npz', train_tf, labeled=True)
    val_set = NpzDataset(INPUT_DIR / 'validation.npz', eval_tf, labeled=True)
    options = dict(batch_size=batch_size, num_workers=workers, pin_memory=DEVICE.type == 'cuda')
    return DataLoader(train_set, shuffle=True, **options), DataLoader(val_set, shuffle=False, **options)

with np.load(INPUT_DIR / 'test.npz') as test_archive:
    print('test keys:', test_archive.files, '| test images:', test_archive['images'].shape)
    assert 'labels' not in test_archive.files, 'Test labels must never be public.'

test keys: ['images', 'ids', 'class_names'] | test images: (51077, 32, 32, 3)


## Supervised baseline: ResNet-18

Start with this fast, native-resolution baseline. Increase `EPOCHS` for a serious run. Try ImageNet pretraining and 224px inputs in the next section.

In [3]:
import time
from tqdm.auto import tqdm

MODEL_NAME = 'resnet18'  # resnet18, resnet50, or vit_b_16
PRETRAINED = False
LINEAR_PROBE = False  # requires PRETRAINED=True
IMAGE_SIZE = 32
BATCH_SIZE = 256
EPOCHS = 50  # starter setting; increase for final experiments

def build_model(name, pretrained, image_size):
    if name == 'resnet18':
        model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
        if image_size == 32 and not pretrained:
            model.conv1, model.maxpool = nn.Conv2d(3, 64, 3, 1, 1, bias=False), nn.Identity()
        model.fc = nn.Linear(model.fc.in_features, 10)
    elif name == 'resnet50':
        model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None)
        model.fc = nn.Linear(model.fc.in_features, 10)
    elif name == 'vit_b_16':
        if image_size != 224: raise ValueError('ViT-B/16 baseline requires IMAGE_SIZE = 224')
        model = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1 if pretrained else None)
        model.heads.head = nn.Linear(model.heads.head.in_features, 10)
    else: raise ValueError(name)
    return model

train_loader, val_loader = loaders(IMAGE_SIZE, PRETRAINED, BATCH_SIZE)
model = build_model(MODEL_NAME, PRETRAINED, IMAGE_SIZE).to(DEVICE)
if LINEAR_PROBE:
    if not PRETRAINED:
        raise ValueError('LINEAR_PROBE=True requires PRETRAINED=True')
    for parameter in model.parameters():
        parameter.requires_grad = False
    head = model.fc if hasattr(model, 'fc') else model.heads
    for parameter in head.parameters():
        parameter.requires_grad = True
optimizer = torch.optim.AdamW(
    (parameter for parameter in model.parameters() if parameter.requires_grad),
    lr=3e-4,
    weight_decay=1e-4,
)
criterion = nn.CrossEntropyLoss()
scaler = GradScaler(DEVICE.type, enabled=DEVICE.type == 'cuda')

def run_epoch(loader, training, desc):
    model.train(training)
    total_loss = total_correct = total = 0
    epoch_start = time.perf_counter()
    progress = tqdm(loader, desc=desc, leave=False)
    for images, labels in progress:
        iter_start = time.perf_counter()
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training), autocast(device_type=DEVICE.type, enabled=DEVICE.type == 'cuda'):
            logits = model(images)
            loss = criterion(logits, labels)
        if training:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        batch_size = len(labels)
        total += batch_size
        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(1) == labels).sum().item()
        iter_time = time.perf_counter() - iter_start
        progress.set_postfix(
            loss=f'{total_loss / total:.4f}',
            acc=f'{total_correct / total:.4f}',
            iter_s=f'{iter_time:.3f}',
        )
    epoch_time = time.perf_counter() - epoch_start
    avg_iter_time = epoch_time / max(len(loader), 1)
    return total_loss / total, total_correct / total, epoch_time, avg_iter_time

best_state, best_accuracy, best_val_loss, best_epoch = None, -1.0, float('inf'), 0
history = []
training_start = time.perf_counter()
for epoch in range(1, EPOCHS + 1):
    train_loss, train_accuracy, train_time, train_iter = run_epoch(
        train_loader, training=True, desc=f'train {epoch}/{EPOCHS}'
    )
    val_loss, val_accuracy, val_time, val_iter = run_epoch(
        val_loader, training=False, desc=f'val {epoch}/{EPOCHS}'
    )
    epoch_time = train_time + val_time
    history.append({
        'epoch': epoch,
        'train_loss': train_loss,
        'train_accuracy': train_accuracy,
        'val_loss': val_loss,
        'val_accuracy': val_accuracy,
        'train_epoch_time_sec': train_time,
        'val_epoch_time_sec': val_time,
        'epoch_time_sec': epoch_time,
        'train_avg_iter_time_sec': train_iter,
        'val_avg_iter_time_sec': val_iter,
    })
    print(
        f'{epoch:02d}: train={train_accuracy:.4f}, val={val_accuracy:.4f}, '
        f'val_loss={val_loss:.4f}, epoch={epoch_time:.2f}s '
        f'(train_iter={train_iter:.3f}s, val_iter={val_iter:.3f}s)'
    )
    if val_accuracy > best_accuracy:
        best_accuracy = val_accuracy
        best_epoch = epoch
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    if val_loss < best_val_loss:
        best_val_loss = val_loss

total_training_time_sec = time.perf_counter() - training_start
model.load_state_dict(best_state)
history_df = pd.DataFrame(history)
history_path = OUTPUT_DIR / 'history.csv'
metrics_path = OUTPUT_DIR / 'training_metrics.csv'
history_df.to_csv(history_path, index=False)
metrics = {
    'best_epoch': best_epoch,
    'best_val_accuracy': best_accuracy,
    'best_val_loss': best_val_loss,
    'total_training_time_sec': total_training_time_sec,
    'mean_epoch_time_sec': float(history_df['epoch_time_sec'].mean()),
    'mean_train_iter_time_sec': float(history_df['train_avg_iter_time_sec'].mean()),
    'mean_val_iter_time_sec': float(history_df['val_avg_iter_time_sec'].mean()),
}
pd.DataFrame([metrics]).to_csv(metrics_path, index=False)
print(
    f'done in {total_training_time_sec:.2f}s | '
    f'best val_acc={best_accuracy:.4f} (epoch {best_epoch}) | '
    f'best val_loss={best_val_loss:.4f}'
)
print(f'saved {history_path}')
print(f'saved {metrics_path}')
history_df

train 1/50:   0%|          | 0/196 [00:00<?, ?it/s]

val 1/50:   0%|          | 0/40 [00:00<?, ?it/s]

01: train=0.4793, val=0.5616, val_loss=1.2572, epoch=42.79s (train_iter=0.199s, val_iter=0.097s)


train 2/50:   0%|          | 0/196 [00:00<?, ?it/s]

val 2/50:   0%|          | 0/40 [00:00<?, ?it/s]

02: train=0.6417, val=0.6669, val_loss=0.9475, epoch=43.93s (train_iter=0.204s, val_iter=0.101s)


... (141 outputs left out)

train 50/50:   0%|          | 0/196 [00:00<?, ?it/s]

val 50/50:   0%|          | 0/40 [00:00<?, ?it/s]

50: train=0.9815, val=0.8967, val_loss=0.4808, epoch=43.30s (train_iter=0.201s, val_iter=0.098s)
done in 2186.45s | best val_acc=0.9027 (epoch 45) | best val_loss=0.3935
saved C:\Users\aimin\Documents\ENS MVA\computer vision\TP3\recvis26_a3\outputs\history.csv
saved C:\Users\aimin\Documents\ENS MVA\computer vision\TP3\recvis26_a3\outputs\training_metrics.csv


,epoch,train_loss,train_accuracy,val_loss,val_accuracy,train_epoch_time_sec,val_epoch_time_sec,epoch_time_sec,train_avg_iter_time_sec,val_avg_iter_time_sec
0,1,1.425637,0.47926,1.257249,0.5616,38.920877,3.871117,42.791994,0.198576,0.096778
1,2,0.996989,0.64166,0.947473,0.6669,39.910608,4.023382,43.933990,0.203626,0.100585
2,3,0.794522,0.72112,0.864158,0.6975,40.521415,4.074194,44.595609,0.206742,0.101855
3,4,0.670778,0.76254,0.824135,0.7182,40.126049,4.077691,44.203740,0.204725,0.101942
4,5,0.590218,0.79280,0.783167,0.7373,40.364511,4.020955,44.385467,0.205941,0.100524
5,6,0.530198,0.81436,0.659800,0.7752,40.156334,4.050914,44.207248,0.204879,0.101273
6,7,0.479702,0.83284,0.665358,0.7826,40.031490,4.061641,44.093131,0.204242,0.101541
7,8,0.442357,0.84586,0.839543,0.7419,40.057071,4.042296,44.099367,0.204373,0.101057
8,9,0.406813,0.85890,0.551886,0.8065,40.043428,3.979654,44.023082,0.204303,0.099491
9,10,0.376549,0.86954,0.493328,0.8385,40.225645,3.940727,44.166373,0.205233,0.098518


### Transfer-learning variants

For an ImageNet-pretrained ResNet use `MODEL_NAME='resnet18'`, `PRETRAINED=True`, and `IMAGE_SIZE=224`. To perform linear probing, freeze all parameters except `model.fc` before creating the optimizer. For ViT, use `MODEL_NAME='vit_b_16'`, `PRETRAINED=True`, and `IMAGE_SIZE=224`.

All external checkpoints and data must be disclosed in the report.

## Generate a Kaggle submission

This uses the held-out test images without attempting to load labels. Upload the resulting `/kaggle/working/submission.csv` file.

In [ ]:
@torch.inference_mode()
def make_submission(model, image_size, pretrained, output_path):
    _, test_tf = transforms_for(image_size, pretrained)
    test_set = NpzDataset(INPUT_DIR / 'test.npz', test_tf, labeled=False)
    test_loader = DataLoader(test_set, batch_size=256, shuffle=False, num_workers=0, pin_memory=DEVICE.type == 'cuda')
    model.eval(); ids, predictions = [], []
    for images, batch_ids in test_loader:
        predictions.extend(model(images.to(DEVICE)).argmax(dim=1).cpu().tolist())
        ids.extend(batch_ids.tolist())
    submission = pd.DataFrame({'id': ids, 'label': predictions})
    if list(submission.columns) != ['id', 'label']:
        raise ValueError('Submission columns must be exactly: id,label')
    if len(submission) != len(test_set) or submission['id'].duplicated().any():
        raise ValueError('Submission must contain each test id exactly once')
    if not np.array_equal(submission['id'].to_numpy(), test_set.ids):
        raise ValueError('Submission ids are not aligned with test.npz')
    if submission['label'].isna().any() or not submission['label'].between(0, 9).all():
        raise ValueError('Predicted labels must be integers from 0 to 9')
    sample = pd.read_csv(INPUT_DIR / 'sample_submission.csv')
    if sample.columns.tolist() != ['id', 'label'] or not np.array_equal(sample['id'].to_numpy(), test_set.ids):
        raise ValueError('sample_submission.csv is not aligned with test.npz')
    submission.to_csv(output_path, index=False)
    return submission

submission = make_submission(model, IMAGE_SIZE, PRETRAINED, OUTPUT_DIR / 'submission.csv')
submission.head(), submission.shape

(   id  label
 0   0      4
 1   1      6
 2   2      1
 3   3      3
 4   4      3,
 (51077, 2))

: 

In [5]:
submission.to_csv(OUTPUT_DIR/ 'submission.csv', index=False)


## Optional: open-source VLMs

You may use open-source vision-language models such as OpenCLIP or Qwen-VL-family models, subject to the assignment policy. Proprietary models and API-only services are prohibited. Record model/checkpoint names, prompts, external data, and all tuning choices in the report.

A zero-shot OpenCLIP baseline can be implemented by encoding one prompt per class and assigning the class with highest image/text similarity. Consider prompt ensembling, but validate carefully: source-domain validation and the hidden target domain differ.

In [ ]:
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Définition autonome de INPUT_DIR et CLASS_NAMES
CLASS_NAMES = ('airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

def _resolve_input_dir():
    if 'RECVIS_INPUT_DIR' in os.environ:
        return Path(os.environ['RECVIS_INPUT_DIR'])
    here = Path.cwd().resolve()
    for candidate in [here / 'data', here, here / 'TP3' / 'recvis26_a3' / 'data', *[p / 'data' for p in here.parents]]:
        if (candidate / 'train.npz').is_file():
            return candidate
    return Path('/kaggle/input/recvis-mva-a3-2026')

INPUT_DIR = _resolve_input_dir()

def sample_3_different_classes(images, labels, rng):
    classes = rng.choice(np.unique(labels), size=3, replace=False)
    indices = [rng.choice(np.where(labels == c)[0]) for c in classes]
    return images[indices], classes

rng = np.random.default_rng()

with np.load(INPUT_DIR / 'train.npz') as train_data:
    train_imgs, train_cls = sample_3_different_classes(train_data['images'], train_data['labels'], rng)

with np.load(INPUT_DIR / 'validation.npz') as val_data:
    val_imgs, val_cls = sample_3_different_classes(val_data['images'], val_data['labels'], rng)

with np.load(INPUT_DIR / 'test.npz') as test_data:
    test_idx = rng.choice(len(test_data['images']), size=3, replace=False)
    test_imgs = test_data['images'][test_idx]
    test_ids = test_data['ids'][test_idx] if 'ids' in test_data else test_idx

fig, axes = plt.subplots(3, 3, figsize=(9, 9))

for j in range(3):
    # Train
    axes[0, j].imshow(train_imgs[j])
    axes[0, j].set_title(f'Train: {CLASS_NAMES[train_cls[j]]} ({train_cls[j]})')
    axes[0, j].axis('off')

    # Validation
    axes[1, j].imshow(val_imgs[j])
    axes[1, j].set_title(f'Val: {CLASS_NAMES[val_cls[j]]} ({val_cls[j]})')
    axes[1, j].axis('off')

    # Test
    axes[2, j].imshow(test_imgs[j])
    axes[2, j].set_title(f'Test (id={test_ids[j]})')
    axes[2, j].axis('off')

plt.tight_layout()
plt.show()